In [1]:
import geopandas as gpd
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
# Load the shapefile containing the coastal profiles
profiles = gpd.read_file(os.path.join("data", "profiles",  "perfiles_playas_limpiosCuadrantes94a120.shp"))
# Create a new column for the profile ID
profiles["profile_id"] = profiles["ID_Perfil"].astype(int)

# Load the shorelines
shorelines = gpd.read_file(os.path.join("data", "shorelines",  "LineasCosta.gpkg"),
                           layer="LineasCostaPlayas")
# Keep only the relevant columns
columns_to_keep = ["ID_LineaCosta", "Mision", "Municipio", "Playa", "Fecha", "Hora", "Cuadrante", "NivelTotal"]
shorelines = shorelines[columns_to_keep + ["geometry"]]
# Filter the shorelines to keep only those where "Cuadrante" is between 94 and 120
shorelines = shorelines[(shorelines["Cuadrante"] >= 94) & (shorelines["Cuadrante"] <= 120)]

# Ensure both GeoDataFrames use the same coordinate reference system (CRS)
profiles = profiles.to_crs(shorelines.crs)
print(f"Profiles CRS: {profiles.crs}")

Profiles CRS: EPSG:25830


In [3]:
# Intersect profiles and shorelines, retaining only point intersections.
profiles_shoreline_intersections = gpd.overlay(
    profiles[["profile_id", "geometry"]],
    shorelines,
    how="intersection",
    keep_geom_type=False,
).explode(index_parts=False, ignore_index=True)
profiles_shoreline_intersections = profiles_shoreline_intersections[
    profiles_shoreline_intersections.geometry.geom_type == "Point"
].copy()

# Shapely's project() returns the along-profile distance in the CRS units (meters here).
profile_geometries = profiles.set_index("profile_id").geometry
profiles_shoreline_intersections["shoreline_position_m"] = (
    profiles_shoreline_intersections.apply(
        lambda row: profile_geometries.loc[row["profile_id"]].project(row.geometry),
        axis=1,
    )
)

# Remove multiple intersections for the same profile and shoreline.
# The shoreline identifier in this dataset is ID_LineaCosta.
profiles_shoreline_intersections = (
    profiles_shoreline_intersections
    .sort_values("shoreline_position_m")
    .drop_duplicates(
        subset=["profile_id", "ID_LineaCosta"],
        keep="first",
    )
)

profiles_shoreline_intersections = gpd.GeoDataFrame(
    profiles_shoreline_intersections[
        ["profile_id"] + columns_to_keep + ["shoreline_position_m", "geometry"]
    ],
    geometry="geometry",
    crs=profiles.crs,
)

# Create a single column for the date and time of the shoreline measurement
profiles_shoreline_intersections["datetime"] = pd.to_datetime(
    profiles_shoreline_intersections["Fecha"].astype(str)
    + " "
    + profiles_shoreline_intersections["Hora"].astype(str),
    #format="%Y%m%d %H%M",
    errors="coerce",
)
# Add the same column to the shorelines GeoDataFrame
shorelines["datetime"] = pd.to_datetime(
    shorelines["Fecha"].astype(str) + " " + shorelines["Hora"].astype(str),
    #format="%Y%m%d %H%M",
    errors="coerce",
)

In [4]:
# Remove Landsat images
profiles_shoreline_intersections = profiles_shoreline_intersections[
    profiles_shoreline_intersections["Mision"] == "S21C"
]

season_map = {
    1: 'Invierno', 2: 'Invierno', 3: 'Primavera',
    4: 'Primavera', 5: 'Primavera', 6: 'Verano',
    7: 'Verano', 8: 'Verano', 9: 'Otoño',
    10: 'Otoño', 11: 'Otoño', 12: 'Invierno'
}
profiles_shoreline_intersections['Estacion'] = profiles_shoreline_intersections['datetime'].dt.month.map(season_map)

# Keep only Verano and Primavera seasons
profiles_shoreline_intersections = profiles_shoreline_intersections[
    profiles_shoreline_intersections['Estacion'].isin(['Primavera', 'Verano'])
]

In [5]:
from utils import SmartTidalFilter

# Initialize the filter with all available parameters explicitly defined
tidal_filter = SmartTidalFilter(
    min_years_span=5.0,                # Minimum temporal span required for a tidal window (in years)
    max_gap_years=2.0,                 # Maximum continuous gap allowed between consecutive data points (in years)
    lambda_1=2.0,                      # Penalty weight for the amplitude of the tidal window (reduces topographic noise)
    lambda_2=1.5,                      # Penalty weight for losing recent data (penalizes windows that drop present years)
    window_sizes=[0.2, 0.4, 0.6, 0.8, 1.0], # List of tidal window widths to test during the optimization search (in meters)
    step_size=0.1,                     # Step size to slide the tidal window across the beach's tidal range (in meters)
    beach_col="Playa",                 # DataFrame column name containing the beach or spatial unit identifier
    tide_col="NivelTotal",             # DataFrame column name containing the sea level/tidal height values
    date_col="datetime"                # DataFrame column name containing the pre-processed datetime objects
)

# Apply the filter to the geospatial dataset
filtered_data = tidal_filter.fit_transform(profiles_shoreline_intersections)

In [6]:
from utils import ShorelineEvolutionAnalyzer

# Initialize the analyzer with a N-day temporal window
analyzer = ShorelineEvolutionAnalyzer(
    window='360D',               # Time-based window
    min_periods=2,               # Needs at least 2 points in those N days to compute a median
    profile_col="profile_id",
    date_col="datetime",         # Must match the datetime column outputted by the tidal filter
    pos_col="shoreline_position_m"
)

# 1. Ejecutar el pipeline completo sobre tus datos previamente filtrados por marea
smoothed_data = analyzer.fit_transform(filtered_data)

# 2. Extraer la tabla final con las tasas de evolución (m/año) y estadísticos
tasas_resultados = analyzer.rates_

# Ver los primeros resultados
display(tasas_resultados.head())

,profile_id,n_observations,rate_ols_m_yr,r_squared,p_value,rate_theilsen_m_yr,theilsen_ci_lower,theilsen_ci_upper,is_significant
0,29897,13,-0.233,0.032,0.5562,-0.018,-1.099,1.567,False
1,29901,13,0.266,0.013,0.7121,0.238,-1.356,1.557,False
2,29902,13,0.054,0.001,0.9277,0.042,-0.998,1.494,False
3,29903,13,-0.209,0.014,0.7031,0.006,-1.540,0.896,False
4,29904,13,-0.104,0.003,0.8683,-0.122,-1.327,1.434,False


In [7]:
# Add a binary field indicating whether the intersection is filtered
smoothed_data['is_filtered'] = 1

# Join relevant attributes from smoothed_data to profiles_shoreline_intersections
profiles_shoreline_intersections = (
    profiles_shoreline_intersections.merge(
        smoothed_data[['ID_LineaCosta', 'profile_id', 'is_filtered', 'shoreline_position_m_smoothed']],
        how='left',
        on=['ID_LineaCosta', 'profile_id']
        )
)
profiles_shoreline_intersections['is_filtered'] = (
    profiles_shoreline_intersections['is_filtered'].fillna(0)
    .astype(int)
)

In [8]:
# Create empty rows for beaches that do not have any filtered intersections,
# so that they are included in the final results
tasas_resultados = (
    tasas_resultados.merge(
        profiles_shoreline_intersections[['Playa', 'profile_id']].drop_duplicates(),
        how='left',
        on='profile_id'
    )
)
tasas_resultados = (
    tasas_resultados.merge(
        profiles_shoreline_intersections[['Playa', 'profile_id']].drop_duplicates(),
        how='right',
        on=['Playa', 'profile_id']
    )
)